# Employment status rolling average: experiment (ticket 2260)

Scores each split and window of the rolling average on known values, then re-scores the best on a temporal hide. Results go to S3 under `RESULTS_PREFIX/RUN_LABEL/`, one file per variant, so an interrupted run resumes where it stopped.

- **Test A (workplace folds):** held-out workplaces are removed from the rolling average inputs, then scored on their own known shares. Every variant runs.
- **Test B (temporal tail hide):** each workplace's last 12 months with known shares are hidden, the pipeline's imputation runs with the variant's rolling average, and the full imputed shares are scored on the hidden values. Runs for the top variants from test A plus the current setup.

**Before running:** set `DATA_BUCKET` and `RESULTS_PREFIX`. **Stop the notebook instance when finished.** Writes only to `RESULTS_PREFIX`.

Shares are scored with every row counting equally, on rows with all 5 known shares. Built to be reusable for other categoricals (such as gender): change the column constants and `SPLITS` in `experiment_utils.py`.

## 0. Setup (SageMaker only)

Run the next two cells once, then restart the kernel.

In [ ]:
%pip install polars==1.44.1 pyarrow boto3

In [ ]:
!rm -rf repo
!git clone --depth 1 --branch 2260-empstat-avg https://github.com/NMDSdevopsServiceAdm/DataEngineering.git repo
!git -C repo rev-parse HEAD

## 1. Parameters

- `DATA_BUCKET`: the bucket holding the cleaned employment status data (read only).
- `RESULTS_PREFIX`: where results are written; `RUN_LABEL` names this run's folder.
- `VARIANT_NAMES`: a list of variant names to run (such as `["current_6mo"]`), or `None` for all.
- `TEST_B_TOP_N`: how many of the best test A variants (lowest MAE) also run test B.
- `LOCAL_ROOT`: set only to dry-run against local files; `None` on SageMaker.
- `REPO_DIR`: where the repo was cloned.

In [ ]:
DATA_BUCKET = "<bucket>"
CLEANED_DATASET = "domain=03_ind_cqc/dataset=02_employment_status_02_clean/"
RESULTS_PREFIX = "s3://<bucket>/sagemaker-notebooks/empstat-rolling-avg/"
RUN_LABEL = "run1"
VARIANT_NAMES = None
TEST_B_TOP_N = 3
N_FOLDS = 5
FOLD_SEED = 42
LOCAL_ROOT = None
REPO_DIR = "repo"

## Setup

Imports, the working copy of the prepared data, and the result locations.

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path(REPO_DIR).resolve()))

import experiments_2260.experiment_utils as x
from polars_utils import utils as polars_utils
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation
from utils.column_names.ind_cqc_pipeline_columns import ModelEvaluationColumns as Eval

LOCAL = LOCAL_ROOT is not None
RESULTS_DIR = (
    str(Path(LOCAL_ROOT) / "results" / RUN_LABEL) if LOCAL else f"{RESULTS_PREFIX.rstrip('/')}/{RUN_LABEL}"
)
SOURCE = str(Path(LOCAL_ROOT) / "cleaned") if LOCAL else f"s3://{DATA_BUCKET}/{CLEANED_DATASET}"
PREPARED_PATH = Path("prepared_2260.parquet")
SELECTED_VARIANTS = [v for v in x.VARIANTS if VARIANT_NAMES is None or v.name in VARIANT_NAMES]
PREDICTED_COLUMNS = x.ROLLING
TRUTH_COLUMNS = [f"{col}_truth" for col in x.KNOWN]

## 2. Prepare the data

Keeps only the columns needed, adds the workplace size band, the short-term imputed (trendline) shares and the 5 workplace folds, then writes the result locally and scans it back. The size band uses the sum of `estimate_filled_posts_by_job_role` per workplace and date, which is present in the cleaned data. Rows are one per workplace, job role and date.

In [ ]:
import projects._03_independent_cqc._02_employment_status.fargate.utils.impute_utils as iUtils

source_lf = pl.scan_parquet(f"{SOURCE}/*.parquet") if LOCAL else polars_utils.scan_parquet(SOURCE)
cleaned_lf = source_lf.select(
    x.LOCATION, x.DATE, x.SERVICE, x.REGION, x.ROLE, x.SIZE_POSTS, *x.KNOWN
)
cleaned_lf = x.add_size_band(cleaned_lf).drop(x.SIZE_POSTS)
cleaned_lf = iUtils.add_short_term_imputed_percentages(
    cleaned_lf,
    extrapolation_period=x.SHORT_TERM_EXTRAPOLATION,
    interpolation_cap_period=x.SHORT_TERM_INTERPOLATION_CAP,
)
cleaned_lf = evaluation.assign_location_folds(cleaned_lf, x.LOCATION, N_FOLDS, FOLD_SEED)
cleaned_lf.sink_parquet(PREPARED_PATH)

prepared_lf = pl.scan_parquet(PREPARED_PATH)
print(prepared_lf.select(pl.len(), pl.col(x.LOCATION).n_unique()).collect())
print(prepared_lf.group_by(x.SIZE_BAND).len().sort(x.SIZE_BAND).collect())

## 3. Test A: held-out workplaces

One file per variant in `test_a/`, skipping any already written. Check `coverage` is 1.0 for every variant: the fallback to a coarser group gives every row a value.

In [ ]:
def variant_columns(df: pl.DataFrame, variant: x.Variant) -> pl.DataFrame:
    return df.with_columns(
        pl.lit(variant.name).alias("variant"),
        pl.lit(variant.split).alias("split"),
        pl.lit(variant.window_months).alias("window_months"),
    )


finished = set(x.list_results("test_a/", RESULTS_DIR, LOCAL))
for variant in SELECTED_VARIANTS:
    name = f"test_a/{variant.name}.parquet"
    if name in finished:
        continue
    pred_lf = x.predict_out_of_fold(prepared_lf, variant, N_FOLDS)
    scores_df = x.score_variant(pred_lf, PREDICTED_COLUMNS, x.KNOWN)
    x.write_result(variant_columns(scores_df, variant), name, RESULTS_DIR, LOCAL)
    print(variant.name, "done")

## 4. Test A summary

Pooled scores per variant. `mean_absolute_error_pp` is lower-is-better; the skill scores are 1 minus the error over a pooled-mean baseline's error, so higher is better. Jumpiness is the mean absolute change between consecutive dates, in share units.

In [ ]:
def load_all(prefix: str) -> pl.DataFrame:
    return pl.concat(
        [x.read_result(name, RESULTS_DIR, LOCAL) for name in x.list_results(prefix, RESULTS_DIR, LOCAL)]
    )


def headline(results_df: pl.DataFrame) -> pl.DataFrame:
    return (
        results_df.filter(
            (pl.col(Eval.level).is_in(["pooled", "coverage", "jumpiness_workplace", "jumpiness_aggregate"]))
            & (pl.col(Eval.metric) != Eval.number_of_rows)
        )
        .with_columns((pl.col(Eval.level) + "_" + pl.col(Eval.metric)).alias("measure"))
        .pivot(on="measure", index="variant", values=Eval.value)
    )


test_a_df = load_all("test_a/")
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(60)
headline(test_a_df).sort(f"pooled_{Eval.mean_absolute_error_pp}")

## 5. Test B: temporal tail hide

Runs the top `TEST_B_TOP_N` variants by test A error, plus the current setup. Each workplace's last 12 months with known shares are hidden. The pipeline's imputation (short-term fill, the variant's rolling average, full imputation) then runs, and the full imputed shares are scored on the hidden values. A variant that loses to the current setup here cannot win.

In [ ]:
ranked = (
    test_a_df.filter(
        (pl.col(Eval.level) == "pooled") & (pl.col(Eval.metric) == Eval.mean_absolute_error_pp)
    )
    .sort(Eval.value)
    .get_column("variant")
    .to_list()
)
variants_by_name = {v.name: v for v in x.VARIANTS}
test_b_variants = [variants_by_name[n] for n in ranked if n != x.CURRENT.name][:TEST_B_TOP_N] + [x.CURRENT]

hidden_lf = x.hide_last_months(prepared_lf)
finished = set(x.list_results("test_b/", RESULTS_DIR, LOCAL))
for variant in test_b_variants:
    name = f"test_b/{variant.name}.parquet"
    if name in finished:
        continue
    imputed_lf = x.run_imputation_with_variant(hidden_lf, variant)
    scores_df = x.score_variant(imputed_lf, x.FULL_IMPUTED, TRUTH_COLUMNS, pl.col(x.HIDDEN))
    x.write_result(variant_columns(scores_df, variant), name, RESULTS_DIR, LOCAL)
    print(variant.name, "done")

## 6. Test B summary

Same measures as test A, on the hidden rows. Coverage and jumpiness are over all rows. Open `compare.ipynb` for the win rule.

In [ ]:
headline(load_all("test_b/")).sort(f"pooled_{Eval.mean_absolute_error_pp}")